<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Transformation — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
# Import the numerical, interpolation and image I/O tools used throughout the lab.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import map_coordinates

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
# Resolve the lab root before loading images or writing outputs.
def find_lab_root(start: Path) -> Path:
    """Find the lab folder without depending on where the notebook was launched.
    
    The notebook may be opened from the lab root or from notebooks/. Instead of
    hard-coding one location, we walk upward until we find a directory containing
    both data/ and notebooks/.
    
    That gives the rest of the notebook one reliable base path to work from."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        # Accept only a parent that contains both the data source and notebook context.
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the Image_Transformation lab root."
    )


LAB_DIR = find_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_FILES = {
    "ascent": DATA_DIR / "ascentB.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "einstein": DATA_DIR / "einstein.png",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
    "peppers": DATA_DIR / "peppers.png",
}

missing = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing, f"Missing input files: {missing}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

## 2. Load and Inspect the Reference Images


In [ ]:
# Load RGB and grayscale views once so all transformations share the same inputs.
images_rgb = {
    name: np.asarray(Image.open(path).convert("RGB"))
    for name, path in IMAGE_FILES.items()
}

images_gray = {
    name: np.asarray(Image.open(path).convert("L"))
    for name, path in IMAGE_FILES.items()
}

for name in IMAGE_FILES:
    rgb = images_rgb[name]
    gray = images_gray[name]

    print(
        f"{name:9s} | "
        f"RGB={str(rgb.shape):16s} "
        f"gray={str(gray.shape):12s} "
        f"dtype={gray.dtype} "
        f"range=[{gray.min()}, {gray.max()}]"
    )

In [ ]:
# Verify all reference images before comparing transformation behavior.
fig, axes = plt.subplots(1, len(images_rgb), figsize=(16, 4))

for ax, (name, image) in zip(axes, images_rgb.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Intensity Transformation Model


In [ ]:
# Establish identity as the baseline every transformation should be compared against.
ascent = images_gray["ascent"]

identity = ascent.copy()

assert np.array_equal(identity, ascent)

print("Identity transformation preserves every pixel:", np.array_equal(identity, ascent))

## 4. Image Negative


In [ ]:
# Use exact 8-bit inversion as the simplest pointwise intensity transform.
negative = 255 - ascent

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(negative, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Negative")

axes[2].plot(np.arange(256), 255 - np.arange(256))
axes[2].set_title("Transformation: s = 255 - r")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].set_xlim(0, 255)
axes[2].set_ylim(0, 255)
axes[2].grid(alpha=0.25)

for ax in axes[:2]:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_negative.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Brightness and Contrast


In [ ]:
# Perform brightness/contrast mapping in float, then clip only at storage boundaries.
def linear_intensity_transform(
    image: np.ndarray,
    gain: float = 1.0,
    offset: float = 0.0,
) -> np.ndarray:
    """Change brightness and contrast with one simple equation.
    
    The model is:
    
        output = gain * input + offset
    
    gain controls contrast, while offset shifts brightness. The computation is done
    in floating point first because intermediate values may fall below 0 or above
    255. Only the final result is clipped back to the valid 8-bit range."""
    transformed = gain * image.astype(np.float32) + offset
    return np.clip(transformed, 0, 255).astype(np.uint8)


# ±50 gives a visible brightness shift while preserving substantial unsaturated content.
brighter = linear_intensity_transform(ascent, gain=1.0, offset=50)
darker = linear_intensity_transform(ascent, gain=1.0, offset=-50)
# Gain 1.5 expands contrast; the negative offset recenters mid-tones after scaling.
higher_contrast = linear_intensity_transform(ascent, gain=1.5, offset=-64)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

examples = [
    ("Original", ascent),
    ("Brightness +50", brighter),
    ("Brightness -50", darker),
    ("Higher contrast", higher_contrast),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_brightness_contrast.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Contrast Stretching


In [ ]:
# Stretch only the occupied intensity range while handling constant images safely.
def contrast_stretch(image: np.ndarray) -> np.ndarray:
    """Use more of the available intensity range.
    
    If an image occupies only a narrow band of gray levels, it can look flat even
    though useful information is present. We linearly map its current minimum and
    maximum to 0 and 255.
    
    A constant image is the special case: there is no range to stretch, so we
    return a stable zero image instead of dividing by zero."""
    image_f = image.astype(np.float32)
    r_min = image_f.min()
    r_max = image_f.max()

    # A constant image has no contrast interval to expand.
    if r_max == r_min:
        return np.zeros_like(image)

    stretched = (image_f - r_min) / (r_max - r_min)
    stretched *= 255.0

    return np.clip(stretched, 0, 255).astype(np.uint8)


# Compress intensities deliberately so contrast stretching has a clear measurable effect.
low_contrast = 90 + (ascent.astype(np.float32) / 255.0) * 75
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

stretched = contrast_stretch(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before stretching")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(stretched, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("Contrast stretched")
axes[1, 0].axis("off")

axes[1, 1].hist(stretched.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After stretching")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_contrast_stretching.png", dpi=300, bbox_inches="tight")
plt.show()

print("Before:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After :", int(stretched.min()), "to", int(stretched.max()))


## 7. Logarithmic Transformation


In [ ]:
# Scale the logarithmic mapping so the 8-bit output range remains valid.
def log_transform(image: np.ndarray) -> np.ndarray:
    """Reveal darker detail by compressing bright intensities.
    
    The logarithm changes rapidly near zero and more slowly at high values. That
    means dark differences are expanded while bright differences are compressed.
    
    The scale factor is chosen so the strongest 8-bit input still maps to 255."""
    image_f = image.astype(np.float32)

    c = 255.0 / np.log1p(255.0)
    transformed = c * np.log1p(image_f)

    return np.clip(transformed, 0, 255).astype(np.uint8)


logged = log_transform(ascent)

r = np.arange(256, dtype=np.float32)
log_curve = (255.0 / np.log1p(255.0)) * np.log1p(r)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(logged, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Log transform")
axes[1].axis("off")

axes[2].plot(r, log_curve)
axes[2].set_title("Log transformation curve")
axes[2].set_xlabel("Input intensity r")
axes[2].set_ylabel("Output intensity s")
axes[2].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_log_transform.png", dpi=300, bbox_inches="tight")
plt.show()


## 8. Gamma / Power-Law Transformation


In [ ]:
# Normalize intensities before applying the power law to keep gamma interpretable.
def gamma_transform(image: np.ndarray, gamma: float) -> np.ndarray:
    """Control tonal emphasis with a power law.
    
    After normalizing intensities to [0, 1]:
    
        gamma < 1  -> brighten darker values
        gamma = 1  -> leave the image unchanged
        gamma > 1  -> darken mid and low values
    
    gamma must stay positive so the mapping remains the standard monotonic power
    law used in image processing."""
    # Positive gamma preserves the standard monotonic power-law mapping.
    if gamma <= 0:
        raise ValueError("gamma must be strictly positive.")

    normalized = image.astype(np.float32) / 255.0
    transformed = normalized ** gamma

    return np.clip(transformed * 255.0, 0, 255).astype(np.uint8)


# Span strong brightening, identity, and strong darkening in one controlled sweep.
gammas = [0.4, 0.7, 1.0, 1.5, 2.2]

fig, axes = plt.subplots(1, len(gammas), figsize=(16, 3.6))

for ax, gamma in zip(axes, gammas):
    transformed = gamma_transform(ascent, gamma)
    ax.imshow(transformed, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"γ = {gamma}")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_gamma_examples.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot transfer curves so gamma behavior is visible independently of one image.
r = np.linspace(0, 1, 256)

fig, ax = plt.subplots(figsize=(6, 4.5))

for gamma in gammas:
    ax.plot(r, r ** gamma, label=f"γ={gamma}")

ax.plot(r, r, linestyle="--", label="identity")
ax.set_title("Gamma / Power-Law Curves")
ax.set_xlabel("Normalized input r")
ax.set_ylabel("Normalized output s")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_gamma_curves.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Histogram Equalization


In [ ]:
def histogram_equalize(image: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Build a contrast mapping from the image itself.
    
    We compute the cumulative histogram and use it as a lookup table. Intensities
    that occupy crowded parts of the histogram are spread out, so the available
    dynamic range is used more effectively.
    
    The mapping is global and monotonic: pixel ordering is preserved, but local
    contrast is not treated independently."""
    histogram = np.bincount(image.ravel(), minlength=256)

    probability = histogram / image.size
    cdf = np.cumsum(probability)

    # Use the normalized CDF as a monotonic intensity mapping.
    mapping = np.round(255 * cdf).astype(np.uint8)
    equalized = mapping[image]

    return equalized, mapping


equalized, equalization_map = histogram_equalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Before equalization")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Original histogram")

axes[1, 0].imshow(equalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After equalization")
axes[1, 0].axis("off")

axes[1, 1].hist(equalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("Equalized histogram")

for ax in [axes[0, 1], axes[1, 1]]:
    ax.set_xlabel("Intensity")
    ax.set_ylabel("Pixel count")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_histogram_equalization.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Inspect the equalization map directly to verify monotonic intensity ordering.
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(np.arange(256), equalization_map)
ax.set_title("Histogram Equalization Mapping")
ax.set_xlabel("Input intensity")
ax.set_ylabel("Mapped intensity")
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_equalization_mapping.png", dpi=300, bbox_inches="tight")
plt.show()


## 10. Compare the Fundamental Intensity Transformations


In [ ]:
# Compare all pointwise transforms on the same source image and display scale.
comparison = [
    ("Original", ascent),
    ("Negative", negative),
    ("Brighter", brighter),
    ("Contrast stretch", contrast_stretch(ascent)),
    ("Log", logged),
    ("Gamma 0.5", gamma_transform(ascent, 0.5)),
    ("Gamma 2.0", gamma_transform(ascent, 2.0)),
    ("Hist. equalized", histogram_equalize(ascent)[0]),
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for ax, (title, image) in zip(axes.ravel(), comparison):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_intensity_transform_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 11. Geometric Transformation Model

In [ ]:
# Start with the geometric idea itself before introducing homogeneous matrices.
def map_coordinates_direct(
    points: np.ndarray,
    scale_x: float = 1.15,
    scale_y: float = 0.85,
    shift_x: float = 35.0,
    shift_y: float = 20.0,
) -> np.ndarray:
    """Move 2-D points with a direct coordinate model.

    A geometric transformation answers a simple question:

        "Where should each point move?"

    Before using matrices, we can write that movement directly:

        x' = scale_x * x + shift_x
        y' = scale_y * y + shift_y

    The scale terms change distances between points. The shift terms move the
    whole geometry without changing its internal shape.

    We intentionally use different x/y scales here so the deformation is easy
    to see. The positive shifts move the transformed rectangle away from the
    original one, which makes the effect visually obvious.

    Sections 12 and 13 will express the same idea more compactly with
    homogeneous coordinates and transformation matrices.
    """
    points = np.asarray(points, dtype=np.float64)

    # Every row must represent exactly one Cartesian point (x, y).
    # Reject anything else now instead of letting a later matrix operation fail cryptically.
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("points must have shape (N, 2).")

    x = points[:, 0]
    y = points[:, 1]

    # Scaling changes relative geometry; translation changes absolute position.
    x_prime = scale_x * x + shift_x
    y_prime = scale_y * y + shift_y

    return np.column_stack((x_prime, y_prime))


# A rectangle is deliberately simple: scaling and translation are immediately visible.
reference_points = np.array(
    [
        [0.0, 0.0],
        [120.0, 0.0],
        [120.0, 80.0],
        [0.0, 80.0],
    ],
    dtype=np.float64,
)

transformed_points = map_coordinates_direct(reference_points)

# The chosen default transform has an exact expected result.
# Checking it proves the coordinate model itself before we move to matrices.
expected_points = np.array(
    [
        [35.0, 20.0],
        [173.0, 20.0],
        [173.0, 88.0],
        [35.0, 88.0],
    ],
    dtype=np.float64,
)

assert transformed_points.shape == reference_points.shape
assert np.all(np.isfinite(transformed_points))
assert np.allclose(transformed_points, expected_points)

print("Original coordinates:")
print(reference_points)

print("\nTransformed coordinates:")
print(np.round(transformed_points, 2))

fig, ax = plt.subplots(figsize=(6, 5))

# Close the polygons by repeating the first point so the rectangle boundary is complete.
closed_reference = np.vstack((reference_points, reference_points[0]))
closed_transformed = np.vstack((transformed_points, transformed_points[0]))

ax.plot(
    closed_reference[:, 0],
    closed_reference[:, 1],
    "o-",
    label="Original coordinates",
)

ax.plot(
    closed_transformed[:, 0],
    closed_transformed[:, 1],
    "o-",
    label="Mapped coordinates",
)

ax.set_title("Direct Coordinate Mapping: (x, y) → (x′, y′)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "10_geometric_coordinate_model.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 12. Homogeneous Coordinates


In [ ]:
# Validate homogeneous transforms first on individual coordinates.
def apply_to_point(matrix: np.ndarray, x: float, y: float) -> np.ndarray:
    """Apply a homogeneous transform to one ordinary 2-D point.
    
    A Cartesian point (x, y) becomes (x, y, 1), the matrix acts on it, then we
    divide by the final homogeneous coordinate to return to ordinary image
    coordinates.
    
    Using one point first makes the matrix convention easy to verify before
    warping an entire image."""
    point = np.array([x, y, 1.0], dtype=np.float64)
    transformed = matrix @ point
    return transformed[:2] / transformed[2]


identity_matrix = np.eye(3)

print("Identity matrix:")
print(identity_matrix)
print("Point (10, 20) ->", apply_to_point(identity_matrix, 10, 20))

## 13. Fundamental Geometric Transformation Matrices


In [ ]:
# Represent all geometric transforms in one homogeneous-coordinate convention.
def translation_matrix(tx: float, ty: float) -> np.ndarray:
    """Create the matrix that shifts every point by (tx, ty).
    
    Translation cannot be represented by a 2x2 linear matrix alone. The extra
    homogeneous coordinate is exactly what lets us place the shift inside the same
    3x3 framework used for rotation, scaling, shear, and reflection."""
    return np.array(
        [[1.0, 0.0, tx],
         [0.0, 1.0, ty],
         [0.0, 0.0, 1.0]]
    )


def scaling_matrix(sx: float, sy: float) -> np.ndarray:
    """Create independent horizontal and vertical scaling.
    
    sx changes x distances and sy changes y distances. Equal values preserve shape;
    different values stretch the image differently along the two axes."""
    return np.array(
        [[sx, 0.0, 0.0],
         [0.0, sy, 0.0],
         [0.0, 0.0, 1.0]]
    )


def rotation_matrix(angle_degrees: float) -> np.ndarray:
    """Create a counter-clockwise rotation about the origin.
    
    The angle is converted from degrees because NumPy trigonometric functions use
    radians. At this stage the pivot is still (0, 0); Section 14 shows how to move
    that pivot to the image center."""
    angle = np.deg2rad(angle_degrees)
    c = np.cos(angle)
    s = np.sin(angle)

    return np.array(
        [[c, -s, 0.0],
         [s,  c, 0.0],
         [0.0, 0.0, 1.0]]
    )


def shear_matrix(kx: float = 0.0, ky: float = 0.0) -> np.ndarray:
    """Create a shear that slants coordinates without rotating them.
    
    kx makes x depend on y, while ky makes y depend on x. Shear preserves straight
    lines but changes angles and shape."""
    return np.array(
        [[1.0, kx, 0.0],
         [ky, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )


def horizontal_reflection_matrix() -> np.ndarray:
    """Mirror x coordinates across the y-axis.
    
    The reflection itself sends positive x values to negative x values. When we use
    it on an image, we therefore combine it with a translation so the reflected
    content is brought back into the visible image frame."""
    return np.array(
        [[-1.0, 0.0, 0.0],
         [0.0, 1.0, 0.0],
         [0.0, 0.0, 1.0]]
    )

## 14. Origin-Centered vs Centered Geometry


In [ ]:
# Re-pivot an origin-based transform so it acts around the image center.
def around_center(matrix, image_shape):
    """Re-pivot a transform around the image center.

    Imagine the transform is a rotation or scaling defined at the origin
    of a coordinate system. For image processing, the origin is the
    top-left corner — but we usually want to rotate or scale around the
    middle of the image.

    So we use a three-step trick:

        - Move the image so its center sits at the origin.
        - Apply the transform there.
        - Move the image back.

    The transform itself does not change; only its pivot changes.

    Mathematically:

        T(c) @ matrix @ T(-c)

    Under the column-vector convention used in this notebook, the
    rightmost matrix acts first.
    """
    height, width = image_shape[:2]

    # Pixel coordinates are zero-indexed, so the geometric center lies halfway
    # between 0 and the final valid coordinate on each axis.
    center_x = (width - 1) / 2
    center_y = (height - 1) / 2

    # First shift the center to the origin; after the transform, shift it back.
    to_origin = translation_matrix(-center_x, -center_y)
    from_origin = translation_matrix(center_x, center_y)

    # Conjugation changes the pivot without changing the underlying transform.
    return from_origin @ matrix @ to_origin


## 15. Forward Mapping vs Inverse Mapping


In [ ]:
def warp_affine(
    image: np.ndarray,
    forward_matrix: np.ndarray,
    output_shape: tuple[int, int] | None = None,
    interpolation_order: int = 1,
    fill_value: float = 0.0,
) -> np.ndarray:
    """Warp an image by asking where every output pixel came from.
    
    A forward transform tells us where a source point wants to go. Using that
    direction directly can leave holes because several source pixels may skip over
    destination positions.
    
    So we reverse the question:
    
        "For this output pixel, which source location should I sample?"
    
    That is inverse mapping. The recovered source coordinate is usually fractional,
    so interpolation estimates its intensity.
    
    Grayscale needs one sampling pass. RGB uses the same geometric coordinates for
    all three channels so color alignment is preserved."""

    # Preserve input dimensions by default so geometry changes are easy to compare.
    if output_shape is None:
        output_shape = image.shape[:2]

    out_h, out_w = output_shape

    # Enumerate destination pixels so inverse mapping produces dense output.
    yy, xx = np.indices((out_h, out_w), dtype=np.float64)
    homogeneous_output = np.stack(
        [xx.ravel(), yy.ravel(), np.ones(xx.size)],
        axis=0,
    )

    # Reverse the transform once. From now on, every output pixel can ask
    # where it should sample from in the original image.
    inverse_matrix = np.linalg.inv(forward_matrix)
    homogeneous_input = inverse_matrix @ homogeneous_output

    x_in = homogeneous_input[0]
    y_in = homogeneous_input[1]

    coordinates = np.vstack([y_in, x_in])

    # A grayscale image has one intensity field, so one sampling pass is enough.
    if image.ndim == 2:
        warped = map_coordinates(
            image.astype(np.float32),
            coordinates,
            order=interpolation_order,
            mode="constant",
            cval=fill_value,
        ).reshape(out_h, out_w)

    # RGB uses the same geometry for every channel; only the sampled intensities differ.
    elif image.ndim == 3:
        channels = []
        for channel_index in range(image.shape[2]):
            sampled = map_coordinates(
                image[..., channel_index].astype(np.float32),
                coordinates,
                order=interpolation_order,
                mode="constant",
                cval=fill_value,
            ).reshape(out_h, out_w)
            channels.append(sampled)

        warped = np.stack(channels, axis=-1)

    else:
        raise ValueError("Expected a 2-D grayscale or 3-D color image.")

    return np.clip(warped, 0, 255).astype(np.uint8)


## 16. Translation


In [ ]:
# Apply translation through the shared affine warp rather than special-case indexing.
einstein = images_gray["einstein"]

# Use a large visible translation while keeping much of the image inside the canvas.
T = translation_matrix(tx=70, ty=35)
translated = warp_affine(
    einstein,
    T,
    # Order 0 preserves nearest-neighbor behavior for integer-like translation/reflection.
interpolation_order=0,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(translated, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translation: tx=70, ty=35")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_translation.png", dpi=300, bbox_inches="tight")
plt.show()

## 17. Rotation


In [ ]:
# Contrast rotation about the image origin with rotation about its center.
# Thirty degrees is large enough to expose pivot choice without making content unrecognizable.
R_origin = rotation_matrix(30)
R_center = around_center(rotation_matrix(30), einstein.shape)

rotated_origin = warp_affine(einstein, R_origin, interpolation_order=1)
rotated_center = warp_affine(einstein, R_center, interpolation_order=1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(rotated_origin, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Rotation about origin")

axes[2].imshow(rotated_center, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotation about image center")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_rotation_origin_center.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Scaling and Resizing


In [ ]:
# Compare uniform and anisotropic scaling around the same geometric pivot.
S_uniform = around_center(
    # 0.65 demonstrates uniform shrinkage while retaining enough structure for comparison.
scaling_matrix(0.65, 0.65),
    einstein.shape,
)

S_nonuniform = around_center(
    # Opposing x/y scales deliberately expose anisotropic shape distortion.
scaling_matrix(1.35, 0.65),
    einstein.shape,
)

scaled_uniform = warp_affine(
    einstein,
    S_uniform,
    # Order 1 uses bilinear interpolation as the default smooth geometric baseline.
interpolation_order=1,
)

scaled_nonuniform = warp_affine(
    einstein,
    S_nonuniform,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(scaled_uniform, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Uniform scale")

axes[2].imshow(scaled_nonuniform, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Non-uniform scale")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Interpolation Comparison


In [ ]:
peppers_rgb = images_rgb["peppers"]

scale_for_interpolation = around_center(
    # Upscale by 1.7 so interpolation differences become visible on the same crop.
scaling_matrix(1.7, 1.7),
    peppers_rgb.shape,
)

nearest = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=0,
)
bilinear = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    interpolation_order=1,
)
bicubic = warp_affine(
    peppers_rgb,
    scale_for_interpolation,
    # Order 3 provides a smoother bicubic-style comparison at higher computational cost.
interpolation_order=3,
)

# Compare interpolation on the same spatial crop.
h, w = peppers_rgb.shape[:2]
crop = (
    slice(h // 3, 2 * h // 3),
    slice(w // 3, 2 * w // 3),
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(peppers_rgb)
axes[0].set_title("Original")

axes[1].imshow(nearest[crop])
axes[1].set_title("Nearest")

axes[2].imshow(bilinear[crop])
axes[2].set_title("Bilinear")

axes[3].imshow(bicubic[crop])
axes[3].set_title("Bicubic")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "14_interpolation_comparison.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 20. Reflection / Flipping


In [ ]:
# Compose reflection with translation so the result remains inside the image frame.
height, width = einstein.shape

F = translation_matrix(width - 1, 0) @ horizontal_reflection_matrix()
reflected = warp_affine(einstein, F, interpolation_order=0)

reflected_numpy = einstein[:, ::-1]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(reflected, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Homogeneous transform")

axes[2].imshow(reflected_numpy, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("NumPy slicing")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_reflection.png", dpi=300, bbox_inches="tight")
plt.show()

print("Two reflection methods identical:", np.array_equal(reflected, reflected_numpy))

## 21. Shear


In [ ]:
# Center the shear to separate shape distortion from unintended translation.
shear_centered = around_center(
    # kx=0.35 creates an obvious horizontal shear without collapsing image geometry.
shear_matrix(kx=0.35, ky=0.0),
    einstein.shape,
)

sheared = warp_affine(
    einstein,
    shear_centered,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(sheared, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Horizontal shear")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_shear.png", dpi=300, bbox_inches="tight")
plt.show()

## 22. Composition of Transformations


In [ ]:
# Reverse transform order explicitly to demonstrate non-commutativity.
# Pair a visible translation with rotation to demonstrate non-commutative composition.
translate = translation_matrix(70, 20)
# A 25-degree centered rotation keeps the composition visually interpretable.
rotate = around_center(rotation_matrix(25), einstein.shape)

translate_then_rotate = rotate @ translate
rotate_then_translate = translate @ rotate

image_a = warp_affine(
    einstein,
    translate_then_rotate,
    interpolation_order=1,
)

image_b = warp_affine(
    einstein,
    rotate_then_translate,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(einstein, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(image_a, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Translate → Rotate")

axes[2].imshow(image_b, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Rotate → Translate")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "17_transformation_order.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## 23. General Affine Transformation


In [ ]:
# Compose several affine operators into one matrix before resampling once.
affine = (
    translation_matrix(35, -10)
    @ around_center(rotation_matrix(-18), einstein.shape)
    @ around_center(shear_matrix(kx=0.18), einstein.shape)
    @ around_center(scaling_matrix(0.88, 1.08), einstein.shape)
)

affine_result = warp_affine(
    images_rgb["einstein"],
    affine,
    interpolation_order=1,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(images_rgb["einstein"])
axes[0].set_title("Original")

axes[1].imshow(affine_result)
axes[1].set_title("Combined affine transform")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_affine_transform.png", dpi=300, bbox_inches="tight")
plt.show()


## 24. Resizing to a New Array Shape


In [ ]:
# Compare interpolation methods at the same target resolution.
source = Image.fromarray(images_rgb["peppers"])

original_width, original_height = source.size

# Downsample by exactly 2x so interpolation methods share one controlled target size.
new_width = original_width // 2
new_height = original_height // 2

resized_nearest = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.NEAREST,
    )
)

resized_bilinear = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BILINEAR,
    )
)

resized_bicubic = np.asarray(
    source.resize(
        (new_width, new_height),
        resample=Image.Resampling.BICUBIC,
    )
)

print("Original shape :", images_rgb["peppers"].shape)
print("Resized shape  :", resized_bilinear.shape)
print(
    "Aspect ratios  :",
    round(original_width / original_height, 4),
    "→",
    round(new_width / new_height, 4),
)

## 25. Validation Checks


In [ ]:
assert np.array_equal(identity, ascent)
assert negative.dtype == np.uint8
assert negative.min() >= 0 and negative.max() <= 255
assert brighter.dtype == np.uint8

# Gamma = 1 is the identity mapping.
gamma_identity = gamma_transform(ascent, 1.0)
assert np.array_equal(gamma_identity, ascent)

# Histogram equalization must preserve intensity ordering.
assert np.all(np.diff(equalization_map.astype(np.int16)) >= 0)

# The direct coordinate model must keep its exact expected geometry.
assert transformed_points.shape == reference_points.shape
assert np.allclose(transformed_points, expected_points)

assert translated.shape == einstein.shape
assert rotated_center.shape == einstein.shape
assert affine_result.shape == images_rgb["einstein"].shape

# Validate geometry with a point whose translated location is known exactly.
known_point = apply_to_point(translation_matrix(12, -5), 10, 20)
assert np.allclose(known_point, [22, 15])

# Reflection must match direct left-right reversal.
assert np.array_equal(reflected, reflected_numpy)

# The selected affine transforms must remain invertible.
assert not np.isclose(np.linalg.det(translation_matrix(10, 20)), 0)
assert not np.isclose(np.linalg.det(rotation_matrix(30)), 0)
assert not np.isclose(np.linalg.det(scaling_matrix(0.5, 1.5)), 0)

print("All image-transformation validation checks passed.")
